In [ ]:
'''
python version 3.10.12
'''


In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [ ]:
from torch.utils.data import DataLoader
# import yaml
from data_utils_SSL import genSpoof_list,Dataset_ASVspoof2019_train,Dataset_ASVspoof2021_eval,Dataset_Eval_eval
from model import Model
# from tensorboardX import SummaryWriter
from core_scripts.startup_config import set_random_seed

2025-01-18 10:59:07 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
/home/ydoit/AIGC/VoiceBench/Learboard/VoiceWukong/aasist2/fairseq/fairseq/tasks/multires_hubert_pretraining.py:154: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  dictionaries = [ (Dictionary.load(f"{label_dir}/dict.{label}.txt") if label is not "" else None ) for label in self.cfg.labels]


In [2]:
import time
def produce_evaluation_file(dataset, model, device, save_path,comment):
    data_loader = DataLoader(dataset, batch_size=24, shuffle=False, drop_last=False)
    num_correct = 0.0
    num_total = 0.0
    model.eval()
    
    variant_list = []
    fname_list = []
    random_list=[]
    src_list=[]
    key_list = []
    score_list = []
    start_time = time.time()
    for batch_x,variants,utt_id,random_token,srcs,labels in data_loader:
        # fname_list = []
        # score_list = []  
        batch_size = batch_x.size(0)
        batch_x = batch_x.to(device)
        
        batch_out = model(batch_x)
        
        batch_score = (batch_out[:, 1]  
                       ).data.cpu().numpy().ravel() 
        # add outputs
        fname_list.extend(utt_id)
        variant_list.extend(variants)
        random_list.extend(random_token)
        src_list.extend(srcs)
        key_list.extend(labels)
        score_list.extend(batch_score.tolist())
    end_time=time.time()
    print(f'{comment} use time : {end_time-start_time} s')
    with open(save_path, 'w') as fh:
        for v,f,r,s,k,cm in zip(variant_list,fname_list,random_list,src_list,key_list,score_list):
            fh.write('{} {} {} {} {} {}\n'.format(v,f,r,s,k, cm))
    # fh.close()   
    print('Scores saved to {}'.format(save_path))

In [ ]:
if __name__ == '__main__':
    '''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
    '''
    database_path='change this to the VoiceWukong dataset path'
    
    batch_size=24
    lr=0.000001
    weight_decay=0.0001
    loss="WCE"
    seed=1234
    model_path="change this to the AASIST2.pth path "

    set_random_seed(seed)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'                  
    print('Device: {}'.format(device))
    
    model = Model(device)
    nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
    model =model.to(device)
    print('nb_params:',nb_params)

    #set Adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)
    
    # if args.model_path:
    model.load_state_dict(torch.load(model_path,map_location=device))
    print('Model loaded : {}'.format(model_path))

    en_eval_set=Dataset_Eval_eval(database_pase=database_path,protocols_file='change this to the path aasist2/SSL_Anti-spoofing/eval_list.txt')
    zh_eval_set=Dataset_Eval_eval(database_pase=database_path,protocols_file='change this to the path aasist2/SSL_Anti-spoofing/zh_eval_list.txt')
    produce_evaluation_file(en_eval_set, model, device, 'change this to the en_eval_score.txt path','en')
    produce_evaluation_file(zh_eval_set, model, device, 'change this to the zh_en_score.txt path','zh')

   
    